In [ ]:
import os, json, math, random, gc, time, ast
from pathlib import Path

import numpy as np
import pandas as pd
import soundfile as sf
import librosa

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision.models import efficientnet_b0

from tqdm import tqdm

print("✅ Imports successful")
print(f"✅ CUDA available: {torch.cuda.is_available()}")

# BirdCLEF 2026 Inference v6 - 2-Model Ensemble (Perch + EfficientNet-B0)
## Strategy:
- Load 5 Perch folds
- Load 5 EfficientNet-B0 folds
- 2-window ensemble (start + middle)
- Average 20 predictions (10 models × 2 windows)
- Apply per-species thresholds

In [ ]:
# === SETUP ===
TEST_AUDIO_DIR = "/kaggle/input/competitions/birdclef-2026/test_soundscapes"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CFG = dict(
    sr=16000,
    n_mels=64,
    n_fft=1024,
    hop=320,
    fmin=60,
    seconds=5,
    batch_size=32,
)

print(f"✅ Config set up")
print(f"✅ Device: {device}")

In [ ]:
# === LOAD SPECIES & THRESHOLDS ===
with open("/kaggle/input/datasets/chiragggg/birdclef-2026-input-model-species/species.json", "r") as f:
    species = json.load(f)

with open("/kaggle/input/datasets/chiragggg/birdclef-2026-input-model-species/optimal_thresholds_v6.json", "r") as f:
    optimal_thresholds = json.load(f)

idx = {lab: i for i, lab in enumerate(species)}
n_classes = len(species)

print(f"✅ Loaded {n_classes} species")
print(f"✅ Loaded per-species thresholds")
print(f"First 10 species: {species[:10]}")

In [ ]:
# === HELPER FUNCTIONS ===
def fixed_length_mono(y, sr, seconds=5):
    target = sr * seconds
    if y.ndim == 2:
        y = y.mean(axis=1)
    if len(y) < target:
        y = np.pad(y, (0, target-len(y)))
    else:
        y = y[:target]
    return y.astype(np.float32)

def logmel_from_wave(wave, sr):
    S = librosa.feature.melspectrogram(
        y=wave, sr=sr, n_fft=CFG["n_fft"], hop_length=CFG["hop"],
        n_mels=CFG["n_mels"], fmin=CFG["fmin"], fmax=sr//2, power=2.0
    )
    S_db = librosa.power_to_db(S, ref=np.max)
    S_min = S_db.min()
    S_max = S_db.max()
    if S_max - S_min < 1e-9:
        S_norm = np.zeros_like(S_db, dtype=np.float32)
    else:
        S_norm = (S_db - S_min) / (S_max - S_min + 1e-9)
    return np.clip(S_norm, 0.0, 1.0).astype(np.float32)

print("✅ Helper functions defined")

In [ ]:
# === PERCH ARCHITECTURE ===
class PerchAudio(nn.Module):
    """Lightweight CNN optimized for bird audio"""
    def __init__(self, n_classes: int, n_mels: int = 64):
        super().__init__()
        
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            
            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            
            nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1))
        )
        
        self.head = nn.Sequential(
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, n_classes)
        )
    
    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        x = self.head(x)
        return x

print("✅ Perch model defined")

In [ ]:
# === EFFICIENTNET-B0 ARCHITECTURE ===
class EfficientNetB0Audio(nn.Module):
    def __init__(self, n_classes: int, n_mels: int = 64):
        super().__init__()
        self.model = efficientnet_b0(weights=None)
        self.model.features[0][0] = nn.Conv2d(1, 32, kernel_size=3, stride=2, padding=1, bias=False)
        in_features = self.model.classifier[1].in_features
        self.model.classifier = nn.Identity()
        self.head = nn.Sequential(
            nn.Linear(in_features, 256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, n_classes)
        )
    
    def forward(self, x):
        features = self.model(x)
        return self.head(features)

print("✅ EfficientNet-B0 model defined")

In [ ]:
# === LOAD ALL TRAINED MODELS ===
print("Loading all 10 trained models (5 folds × 2 architectures)...")

perch_models = []
efficientnet_models = []

for fold_idx in range(5):
    # Load Perch
    perch_model = PerchAudio(n_classes=n_classes, n_mels=CFG["n_mels"]).to(device)
    perch_model.load_state_dict(torch.load(f"/kaggle/input/datasets/chiragggg/birdclef-2026-input-model-species/perch_v6_fold_{fold_idx}.pt"))
    perch_model.eval()
    perch_models.append(perch_model)
    
    # Load EfficientNet-B0
    eff_model = EfficientNetB0Audio(n_classes=n_classes, n_mels=CFG["n_mels"]).to(device)
    eff_model.load_state_dict(torch.load(f"/kaggle/input/datasets/chiragggg/birdclef-2026-input-model-species/efficientnet_v6_fold_{fold_idx}.pt"))
    eff_model.eval()
    efficientnet_models.append(eff_model)

print(f"✅ Loaded {len(perch_models)} Perch folds")
print(f"✅ Loaded {len(efficientnet_models)} EfficientNet-B0 folds")

In [ ]:
# === TEST AUDIO DATASET (2-WINDOW STRATEGY) ===
class TestAudioDataset(Dataset):
    def __init__(self, audio_path, cfg):
        try:
            y, sr0 = sf.read(audio_path, always_2d=False)
        except:
            print(f"Failed to load: {audio_path}")
            y, sr0 = np.zeros(cfg["sr"] * cfg["seconds"]), cfg["sr"]
        
        if sr0 != cfg["sr"]:
            y = librosa.resample(y, orig_sr=sr0, target_sr=cfg["sr"])
        
        y = fixed_length_mono(y, cfg["sr"], cfg["seconds"])
        self.mel = logmel_from_wave(y, cfg["sr"])
        self.win_frames = int(cfg["seconds"] * cfg["sr"] / cfg["hop"]) + 1
    
    def __len__(self):
        # Two windows per audio
        return 2
    
    def __getitem__(self, idx):
        T = self.mel.shape[1]
        W = self.win_frames
        
        # Two-window: start and middle
        if idx == 0:
            start = 0
        else:
            start = max(0, (T - W) // 2)
        
        if T <= W:
            pad = np.zeros((self.mel.shape[0], W - T), dtype=np.float32)
            mel = np.concatenate([self.mel, pad], axis=1)
        else:
            mel = self.mel[:, start:start + W]
        
        mel = np.clip(mel, 0.0, 1.0).astype(np.float32)
        x = torch.from_numpy(mel)[None, ...]
        return x.float()

print("✅ Test dataset class defined")

In [ ]:
# === PREDICTION FUNCTION ===
def get_predictions_for_audio(audio_path):
    """Get ensemble predictions from 10 models × 2 windows"""
    
    dataset = TestAudioDataset(audio_path, CFG)
    loader = DataLoader(dataset, batch_size=1, shuffle=False, num_workers=0)
    
    all_preds = np.zeros((n_classes,), dtype=np.float32)
    
    with torch.no_grad():
        for window_idx, x in enumerate(loader):
            x = x.to(device)
            
            # Get predictions from all Perch folds
            for perch_model in perch_models:
                logits = perch_model(x)
                probs = torch.sigmoid(logits).cpu().numpy()[0]
                all_preds += probs
            
            # Get predictions from all EfficientNet folds
            for eff_model in efficientnet_models:
                logits = eff_model(x)
                probs = torch.sigmoid(logits).cpu().numpy()[0]
                all_preds += probs
    
    # Average over all windows and models (2 windows × 5 folds × 2 architectures = 20 predictions)
    ensemble_preds = all_preds / (2 * 5 * 2)
    
    return ensemble_preds

print("✅ Prediction function defined")

In [ ]:
# === GENERATE PREDICTIONS ===
test_dir = Path(TEST_AUDIO_DIR)
test_files = sorted(test_dir.glob("*.ogg"))

print(f"Found {len(test_files)} test files")

predictions = {}

for audio_path in tqdm(test_files, desc="Generating predictions"):
    filename = audio_path.stem
    ensemble_preds = get_predictions_for_audio(str(audio_path))
    predictions[filename] = ensemble_preds

print(f"\n✅ Generated predictions for {len(predictions)} test files")

In [ ]:
# === APPLY PER-SPECIES THRESHOLDS & CREATE SUBMISSION ===
submission_data = []

for filename, preds in predictions.items():
    row = {"row_id": filename}
    
    for j, sp in enumerate(species):
        threshold = optimal_thresholds.get(sp, 0.5)
        prediction = 1 if preds[j] >= threshold else 0
        row[sp] = prediction
    
    submission_data.append(row)

submission_df = pd.DataFrame(submission_data)

print(f"\n✅ Submission DataFrame created")
print(f"Shape: {submission_df.shape}")
print(f"\nFirst 5 rows (first 10 columns):")
print(submission_df.iloc[:5, :10])

In [ ]:
# === SAVE SUBMISSION ===
submission_df.to_csv("/kaggle/working/submission_v6_perch_efficientnet.csv", index=False)

print(f"✅ Submission saved to: /kaggle/working/submission_v6_perch_efficientnet.csv")
print(f"\n📊 SUBMISSION SUMMARY:")
print(f"  Total predictions: {len(submission_df)}")
print(f"  Total species: {len(species)}")
print(f"  Avg positive predictions per test: {submission_df[species].sum(axis=1).mean():.2f}")
print(f"\n✅ INFERENCE COMPLETE!")
print(f"\n📈 ENSEMBLE STRATEGY:")
print(f"  - 5 Perch folds")
print(f"  - 5 EfficientNet-B0 folds")
print(f"  - 2 windows per test audio (start and middle)")
print(f"  - Total: 20 predictions averaged per test file (vs 30 in v5)")
print(f"  - Per-species optimal thresholds applied")